# 14.3 Calculus detection: YOLO26n comparison

Compare with YOLOv8n version 3 using the same source, positive-only data preparation, split, input size, seed and nominal training budget. The Ultralytics version and YOLO training recipes differ, so this is a practical model comparison rather than an architecture-only ablation.

## 1. Install and identify the dataset

In [ ]:
%pip install -q ultralytics==8.4.183

In [ ]:
from pathlib import Path
import os, random, shutil, hashlib, json
from collections import Counter
from ultralytics import YOLO
import torch
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
files = list(Path("/kaggle/input").rglob("data.yaml"))
print("Dataset YAML candidates:", files)
assert len(files) == 1, "Attach exactly the Singh868 dataset"
source = files[0].parent
print(files[0].read_text())
assert "0: calculus" in files[0].read_text().lower()

## 2. Build single-class training data

Retain calculus-positive images only, using their original calculus boxes. Non-calculus annotations are not treated as negatives. Exclude invalid annotations and exact train-validation duplicates. Test images are never opened.

In [ ]:
target = Path("/tmp/calculus_data")
counts = Counter()
train_hashes = set()
for split, out_split in (("train", "train"), ("valid", "val")):
    (target / "images" / out_split).mkdir(parents=True, exist_ok=True)
    (target / "labels" / out_split).mkdir(parents=True, exist_ok=True)
    for img in sorted((source / split / "images").iterdir()):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
        label = source / split / "labels" / (img.stem + ".txt")
        if not label.exists(): counts[(split,"missing_label")] += 1; continue
        rows = []
        invalid = False
        for line in label.read_text().splitlines():
            try:
                values = list(map(float, line.split()))
                if len(values) != 5 or int(values[0]) != values[0] or int(values[0]) not in range(5): raise ValueError()
                _, x, y, w, h = values
                if not (0 <= x-w/2 and x+w/2 <= 1 and 0 <= y-h/2 and y+h/2 <= 1 and w > 0 and h > 0): raise ValueError()
                if int(values[0]) == 0: rows.append(line)
            except ValueError: invalid = True
        if invalid: counts[(split,"invalid")] += 1; continue
        if not rows: continue
        digest = hashlib.sha256(img.read_bytes()).hexdigest()
        if split == "valid" and digest in train_hashes: counts[(split,"cross_split_duplicate")] += 1; continue
        if split == "train": train_hashes.add(digest)
        shutil.copyfile(img, target / "images" / out_split / img.name)
        (target / "labels" / out_split / (img.stem + ".txt")).write_text("\n".join(rows)+"\n")
        counts[(split,"images")] += 1
        counts[(split,"boxes")] += len(rows)
print(dict(counts))
assert counts[("train","images")] > 0 and counts[("valid","images")] > 0
(target / "data.yaml").write_text(f"path: {target}\ntrain: images/train\nval: images/val\nnames:\n  0: calculus\n")

## 3. Inspect source labels

Show an image and its original boxes before optimization.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
img_path = sorted((target / "images/train").iterdir())[0]
im = Image.open(img_path).convert("RGB")
draw = ImageDraw.Draw(im)
for line in (target / "labels/train" / (img_path.stem + ".txt")).read_text().splitlines():
    _, x, y, w, h = map(float, line.split())
    draw.rectangle(((x-w/2)*im.width, (y-h/2)*im.height, (x+w/2)*im.width, (y+h/2)*im.height), outline="red", width=3)
plt.figure(figsize=(9,7)); plt.imshow(im); plt.axis("off"); plt.show()

## 4. Train YOLO26n

One GPU worker. Maximum 60 epochs, early stopping and a 1-hour training bound. Save final checkpoints. Compare mobile latency separately.

In [ ]:
model = YOLO("yolo26n.pt")
results = model.train(data=str(target / "data.yaml"), epochs=60, patience=12, imgsz=640, batch=16, device=0, workers=2, seed=42, deterministic=True, project="/kaggle/working/runs", name="calculus_yolo26n", exist_ok=True, save=True, plots=True, amp=True, time=1.0)
print("Training complete:", results.save_dir)

## 5. Review validation metrics and checkpoints

This positive-only validation set does not measure false positives on confirmed negative images. The source split may share underlying cases or near-duplicates.

In [ ]:
print("Best checkpoint:", Path("/kaggle/working/runs/calculus_yolo26n/weights/best.pt").exists())
print("Last checkpoint:", Path("/kaggle/working/runs/calculus_yolo26n/weights/last.pt").exists())
print(results.results_dict)

## 6. Compare with the previous run

The historical YOLOv8n v3 best epoch reported 75.471% mAP50 and 42.435% mAP50-95. Source-only positive validation does not measure false positives on negative images. Check the observed dataset counts and model configuration before comparison.

In [ ]:
print("YOLOv8n v3 reference: mAP50=0.75471, mAP50-95=0.42435")
print("YOLO26n run:", results.results_dict)